# AgroSight — PlantVillage Multi-SKU Classifier

**Companion to `AgroSight_CV_Complete.ipynb` (Beans pilot)** — keeps beans results, adds multi-crop scale.

| Notebook | Dataset | Role |
|----------|---------|------|
| `AgroSight_CV_Complete.ipynb` | Beans (~1.3k) | Pilot / pipeline validation |
| **This notebook** | PlantVillage (6 SKUs, ~5k subset) | Production multi-SKU model |

## Colab setup
1. **Runtime → T4 GPU**
2. Run **Cell 1** → **Restart session** → skip Cell 1
3. Run All from Cell 2 (~40–60 min; **first run downloads ~800MB** `data.zip` + extracts images)

**Outputs:** `07–10_*.png` in `dashboard/assets/`, model in `models/agrosight_plantvillage_*`, `sku_catalog.json`

In [ ]:
# CELL 1 — INSTALL
!pip install -q datasets pillow scikit-learn seaborn matplotlib pandas huggingface_hub
print('DONE — Restart session, skip this cell, run Cell 2+')

In [ ]:
# CELL 2 — IMPORTS + LOAD PLANTVILLAGE
import warnings, time, os, json
from pathlib import Path
from collections import defaultdict
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import tensorflow as tf
from PIL import Image
from huggingface_hub import hf_hub_download
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2, EfficientNetB0, ResNet50V2, InceptionV3
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as prep_mobilenet
from tensorflow.keras.applications.efficientnet import preprocess_input as prep_efficientnet
from tensorflow.keras.applications.resnet_v2 import preprocess_input as prep_resnet
from tensorflow.keras.applications.inception_v3 import preprocess_input as prep_inception
from sklearn.metrics import classification_report, confusion_matrix, f1_score

tf.random.set_seed(42)
np.random.seed(42)

NOTEBOOK_DIR = Path.cwd()
ASSETS_DIR   = NOTEBOOK_DIR / 'dashboard' / 'assets'
DATA_DIR     = NOTEBOOK_DIR / 'dashboard' / 'data'
MODELS_DIR   = NOTEBOOK_DIR / 'models'
for d in [ASSETS_DIR, DATA_DIR, MODELS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

BG, CARD, GRID, TICK = '#0a0f1e', '#111827', '#1e293b', '#94a3b8'
DEFECT_CLASSES = ['HEALTHY', 'SURFACE_DEFECT', 'BLIGHT_MOLD']
NUM_CLASSES = 3

# 6 AgroSight SKUs aligned to rural food processing
SKU_MAP = {
    'Corn_(maize)':   {'sku': 'MAIZE_GRAIN',    'category': 'Grains',      'icon': 'corn'},
    'Soybean':        {'sku': 'SOYBEAN_PULSE',  'category': 'Pulses',      'icon': 'soybean'},
    'Pepper,_bell':   {'sku': 'PEPPER_SPICE',   'category': 'Spices',      'icon': 'pepper'},
    'Tomato':         {'sku': 'TOMATO_VEG',     'category': 'Vegetables',  'icon': 'tomato'},
    'Potato':         {'sku': 'POTATO_VEG',     'category': 'Vegetables',  'icon': 'potato'},
    'Apple':          {'sku': 'APPLE_FRUIT',    'category': 'Fruits',      'icon': 'apple'},
}
SELECTED_CROPS = set(SKU_MAP.keys())
MAX_PER_CROP_TRAIN = 800
MAX_PER_CROP_TEST  = 120

# Download color images from zip (HF 'default' config has paths only, not pixels)
import zipfile

def path_to_crop(rel_path):
    folder = rel_path.replace('\\', '/').split('/')[-2]
    return folder.split('___')[0]

def path_to_class_name(rel_path):
    return rel_path.replace('\\', '/').split('/')[-2]

def map_to_defect(name):
    n = name.lower()
    if 'healthy' in n:
        return 0
    blight_kw = ['blight', 'rot', 'rust', 'mildew', 'mold', 'esca', 'greening', 'curl']
    if any(k in n for k in blight_kw):
        return 2
    return 1

def read_paths(txt_file):
    with open(txt_file) as f:
        return [l.strip() for l in f if l.strip()]

def resolve_image(rel_path):
    rel = rel_path.replace('\\', '/')
    for base in [PV_ROOT, PV_ROOT / 'data']:
        p = base / rel
        if p.exists():
            return p
    raise FileNotFoundError(rel_path)

def load_image(rel_path):
    return np.array(Image.open(resolve_image(rel_path)).convert('RGB'), dtype=np.uint8)

def subsample_paths(paths, max_per_crop):
    by_crop = defaultdict(list)
    for p in paths:
        crop = path_to_crop(p)
        if crop in SELECTED_CROPS:
            by_crop[crop].append(p)
    out = []
    for crop in sorted(SELECTED_CROPS):
        out.extend(by_crop[crop][:max_per_crop])
    return out

print('Downloading PlantVillage (~800MB images, one-time)...')
zip_path = hf_hub_download(repo_id='mohanty/PlantVillage', filename='data.zip', repo_type='dataset')
train_txt = hf_hub_download(repo_id='mohanty/PlantVillage', filename='splits/color_train.txt', repo_type='dataset')
test_txt  = hf_hub_download(repo_id='mohanty/PlantVillage', filename='splits/color_test.txt', repo_type='dataset')

PV_ROOT = Path('/content/pv_data') if Path('/content').exists() else NOTEBOOK_DIR / 'pv_data'
if not (PV_ROOT / 'raw').exists() and not (PV_ROOT / 'data' / 'raw').exists():
    PV_ROOT.mkdir(parents=True, exist_ok=True)
    print('Extracting data.zip...')
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(PV_ROOT)

all_train = [p for p in read_paths(train_txt) if 'color' in p.replace('\\', '/')]
all_test  = [p for p in read_paths(test_txt)  if 'color' in p.replace('\\', '/')]
print(f'Color split lists: train={len(all_train)}, test={len(all_test)}')

train_paths = subsample_paths(all_train, MAX_PER_CROP_TRAIN)
test_paths  = subsample_paths(all_test,  MAX_PER_CROP_TEST)
rng = np.random.default_rng(42)
rng.shuffle(train_paths)
test_crops = [path_to_crop(p) for p in test_paths]

n_classes = len({path_to_class_name(p) for p in train_paths + test_paths})
print(f'AgroSight subset: train={len(train_paths)}, test={len(test_paths)}')
print(f'Disease classes in subset: {n_classes}')
for crop in sorted(SELECTED_CROPS):
    tr = sum(1 for p in train_paths if path_to_crop(p) == crop)
    te = sum(1 for p in test_paths if path_to_crop(p) == crop)
    print(f'  {SKU_MAP[crop]["sku"]:18s} train={tr:4d}  test={te:3d}')

sku_catalog = {
    'defect_classes': DEFECT_CLASSES,
    'skus': [{'crop_key': k, **v} for k, v in SKU_MAP.items()],
    'source': 'mohanty/PlantVillage (color, zip)',
    'train_images': len(train_paths),
    'test_images': len(test_paths),
    'disease_classes_in_subset': n_classes,
}
with open(DATA_DIR / 'sku_catalog.json', 'w') as f:
    json.dump(sku_catalog, f, indent=2)
print('Saved sku_catalog.json')

In [ ]:
# CELL 3 — BUILD TF DATASETS (generator = low RAM)
IMG_SIZE, BATCH_SIZE, AUTOTUNE = 224, 32, tf.data.AUTOTUNE

augment = keras.Sequential([
    layers.RandomFlip('horizontal_and_vertical'),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.1),
    layers.RandomBrightness(0.1, value_range=(0, 255)),
    layers.RandomContrast(0.1, value_range=(0, 255)),
], name='augmentation')

output_sig = (
    tf.TensorSpec(shape=(None, None, 3), dtype=tf.uint8),
    tf.TensorSpec(shape=(), dtype=tf.int32),
)

def make_gen(path_list):
    def _gen():
        for p in path_list:
            yield load_image(p), map_to_defect(path_to_class_name(p))
    return _gen

ds_train = tf.data.Dataset.from_generator(make_gen(train_paths), output_signature=output_sig)
ds_test  = tf.data.Dataset.from_generator(make_gen(test_paths),  output_signature=output_sig)

def preprocess(image, label):
    image = tf.cast(image, tf.float32)
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    return image, label

def preprocess_train(image, label):
    image, label = preprocess(image, label)
    image = augment(image, training=True)
    return image, label

train_ds = ds_train.map(preprocess_train, AUTOTUNE).shuffle(2000).batch(BATCH_SIZE).prefetch(AUTOTUNE)
test_ds  = ds_test.map(preprocess, AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)
raw_ds   = ds_train.map(preprocess, AUTOTUNE).batch(6).take(1)
print('Pipelines ready')

In [ ]:
# CELL 4 — MULTI-SKU SAMPLE VISUALISATION
# One sample per SKU (6 crops)
shown, sku_samples = {}, []
for p in train_paths:
    crop = path_to_crop(p)
    if crop in shown:
        continue
    shown[crop] = True
    sku_samples.append((crop, load_image(p), map_to_defect(path_to_class_name(p))))
    if len(shown) == 6:
        break

fig, axes = plt.subplots(2, 6, figsize=(18, 6))
fig.patch.set_facecolor(BG)
fig.suptitle('AgroSight — Multi-SKU PlantVillage Samples (6 crops, 3 defect classes)',
             color='white', fontsize=14, fontweight='bold')

for i, (crop, arr, lbl) in enumerate(sku_samples):
    sku = SKU_MAP[crop]['sku']
    axes[0, i].imshow(arr)
    axes[0, i].set_title(sku.replace('_', ' '), color='#7dd3fc', fontsize=8)
    axes[0, i].axis('off')
    axes[1, i].imshow(arr)
    axes[1, i].set_title(DEFECT_CLASSES[lbl], color='#a78bfa', fontsize=8)
    axes[1, i].axis('off')

plt.tight_layout()
out = ASSETS_DIR / '07_pv_multi_sku_samples.png'
plt.savefig(out, dpi=120, bbox_inches='tight', facecolor=BG)
plt.close(fig)
print(f'Saved: {out}')

In [ ]:
# CELL 5 — BUILD 5 MODELS (same architecture as Beans notebook)
def build_baseline(nc, sz=224):
    return keras.Sequential([
        keras.Input((sz, sz, 3)), layers.Rescaling(1./255),
        layers.Conv2D(32,3,padding='same',activation='relu'), layers.BatchNormalization(), layers.MaxPooling2D(),
        layers.Conv2D(64,3,padding='same',activation='relu'), layers.BatchNormalization(), layers.MaxPooling2D(),
        layers.Conv2D(128,3,padding='same',activation='relu'), layers.BatchNormalization(), layers.MaxPooling2D(),
        layers.GlobalAveragePooling2D(), layers.Dropout(0.4),
        layers.Dense(256, activation='relu'), layers.Dropout(0.3),
        layers.Dense(nc, activation='softmax'),
    ], name='Baseline_CNN')

def build_transfer(fn, name, prep, nc, sz=224):
    base = fn(include_top=False, weights='imagenet', input_shape=(sz,sz,3))
    base.trainable = False
    inp = keras.Input((sz,sz,3))
    x = layers.Lambda(prep)(inp)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(nc, activation='softmax')(x)
    return keras.Model(inp, out, name=name), base

all_models, all_bases = {}, {}
all_models['Baseline_CNN'] = build_baseline(NUM_CLASSES)
all_bases['Baseline_CNN'] = None
for fn, nm, prep in [
    (MobileNetV2,'MobileNetV2',prep_mobilenet),
    (EfficientNetB0,'EfficientNetB0',prep_efficientnet),
    (ResNet50V2,'ResNet50V2',prep_resnet),
    (InceptionV3,'InceptionV3',prep_inception),
]:
    m, b = build_transfer(fn, nm, prep, NUM_CLASSES)
    all_models[nm], all_bases[nm] = m, b

for n, m in all_models.items():
    print(f'{n:20s} {m.count_params():>12,} params')

In [ ]:
# CELL 6 — TRAIN 5 MODELS (Phase 1)
GPUS = tf.config.list_physical_devices('GPU')
EPOCHS, LR = (8 if GPUS else 5), 1e-3
print(f'Phase 1: {EPOCHS} epochs')

def cbs():
    return [
        keras.callbacks.EarlyStopping('val_accuracy', patience=4, restore_best_weights=True, verbose=0),
        keras.callbacks.ReduceLROnPlateau('val_loss', factor=0.5, patience=2, min_lr=1e-7, verbose=0),
    ]

histories, metrics = {}, {}
for name, model in all_models.items():
    print(f'\n>> {name}')
    model.compile(optimizer=keras.optimizers.Adam(LR), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    t0 = time.time()
    h = model.fit(train_ds, validation_data=test_ds, epochs=EPOCHS, callbacks=cbs(), verbose=1)
    _, acc = model.evaluate(test_ds, verbose=0)
    y_true, y_pred = [], []
    for imgs, lbls in test_ds:
        p = model(imgs, training=False)
        y_true.extend(lbls.numpy()); y_pred.extend(np.argmax(p.numpy(), 1))
    f1 = f1_score(y_true, y_pred, average='weighted')
    sb, _ = next(iter(test_ds.take(1)))
    t1 = time.time()
    for _ in range(5): _ = model(sb, training=False)
    inf = (time.time()-t1)/(5*BATCH_SIZE)*1000
    histories[name] = h
    metrics[name] = {'Accuracy':acc,'F1':f1,'Inference_ms':inf,'Time_min':(time.time()-t0)/60}
    print(f'   Acc={acc*100:.1f}%  F1={f1:.3f}  {inf:.1f}ms/img')
print('\nPhase 1 done')

In [ ]:
# CELL 7 — MODEL COMPARISON CHART
df = pd.DataFrame(metrics).T.reset_index()
df.columns = ['Model','Accuracy','F1','Inference_ms','Time_min']
df = df.sort_values('Accuracy', ascending=False)
COLORS = ['#6366f1','#06b6d4','#10b981','#f59e0b','#f43f5e']

fig = plt.figure(figsize=(18, 10))
fig.patch.set_facecolor(BG)
gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.3)
names = df['Model'].tolist()

for i, (metric, title, ax_idx) in enumerate([
    ('Accuracy','Accuracy (%)', (0,0)), ('F1','F1 Score', (0,1)), ('Inference_ms','Inference ms', (1,0))]):
    ax = fig.add_subplot(gs[ax_idx[0], ax_idx[1]])
    ax.set_facecolor(CARD)
    vals = df[metric]*100 if metric=='Accuracy' else df[metric]
    bars = ax.bar(names, vals, color=COLORS, width=0.55)
    ax.set_title(f'PlantVillage — {title}', color='white', fontweight='bold')
    ax.tick_params(colors=TICK); ax.set_xticklabels(names, rotation=25, ha='right', fontsize=7)
    for b,v in zip(bars, vals):
        ax.text(b.get_x()+b.get_width()/2, b.get_height(), f'{v:.1f}', ha='center', color='white', fontsize=8)

ax4 = fig.add_subplot(gs[1,1])
ax4.set_facecolor(CARD)
for i,(nm,h) in enumerate(histories.items()):
    va = h.history.get('val_accuracy',[])
    ax4.plot(range(1,len(va)+1), [v*100 for v in va], color=COLORS[i], lw=2, label=nm)
ax4.set_title('Val Accuracy per Epoch', color='white', fontweight='bold')
ax4.tick_params(colors=TICK); ax4.legend(fontsize=7, facecolor=CARD, labelcolor='white')

fig.suptitle('PlantVillage Multi-SKU — 5-Model Comparison', color='white', fontsize=16, fontweight='bold')
out = ASSETS_DIR / '08_pv_model_comparison.png'
plt.savefig(out, dpi=130, bbox_inches='tight', facecolor=BG)
plt.close(fig)
print(f'Saved: {out}')

In [ ]:
# CELL 8 — SELECT WINNER + FINE-TUNE
sc = df.copy()
for col, inv in [('Accuracy',False),('F1',False),('Inference_ms',True)]:
    mn, mx = sc[col].min(), sc[col].max()
    n = (sc[col]-mn)/(mx-mn+1e-9)
    sc[col+'_n'] = 1-n if inv else n
sc['Score'] = 0.6*sc['Accuracy_n'] + 0.3*sc['F1_n'] + 0.1*sc['Inference_ms_n']
sc = sc.sort_values('Score', ascending=False)
BEST = sc.iloc[0]['Model']
best_model = all_models[BEST]
print(f'Winner: {BEST} (score={sc.iloc[0]["Score"]:.4f})')

EPOCHS_FT, LR_FT = (5 if GPUS else 3), 1e-5
base = all_bases[BEST]
if base is not None:
    for layer in base.layers[-20:]: layer.trainable = True
    best_model.compile(optimizer=keras.optimizers.Adam(LR_FT),
        loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    best_model.fit(train_ds, validation_data=test_ds, epochs=EPOCHS_FT, callbacks=cbs(), verbose=1)
    _, ft_acc = best_model.evaluate(test_ds, verbose=0)
    metrics[BEST]['Accuracy_FT'] = ft_acc
    print(f'After fine-tune: {ft_acc*100:.2f}%')
else:
    print('Baseline CNN — skip fine-tune')

In [ ]:
# CELL 9 — CONFUSION MATRIX + PER-SKU ACCURACY
y_true, y_pred = [], []
for imgs, lbls in test_ds:
    p = best_model(imgs, training=False)
    y_true.extend(lbls.numpy())
    y_pred.extend(np.argmax(p.numpy(), 1))

print(classification_report(y_true, y_pred, target_names=DEFECT_CLASSES))

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(7,5))
fig.patch.set_facecolor(BG); ax.set_facecolor(CARD)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=DEFECT_CLASSES,
            yticklabels=DEFECT_CLASSES, ax=ax, linewidths=0.5)
ax.set_title(f'{BEST} — PlantVillage Confusion Matrix', color='white', fontweight='bold')
plt.tight_layout()
plt.savefig(ASSETS_DIR/'09_pv_confusion_matrix.png', dpi=130, bbox_inches='tight', facecolor=BG)
plt.close()

# Per-SKU breakdown
sku_acc = {}
for crop in SELECTED_CROPS:
    idxs = [i for i, c in enumerate(test_crops) if c == crop]
    if idxs:
        correct = sum(1 for i in idxs if y_true[i]==y_pred[i])
        sku_acc[SKU_MAP[crop]['sku']] = correct/len(idxs)*100

fig, ax = plt.subplots(figsize=(10,4))
fig.patch.set_facecolor(BG); ax.set_facecolor(CARD)
skus = list(sku_acc.keys()); accs = list(sku_acc.values())
ax.barh(skus, accs, color='#6366f1')
ax.set_xlabel('Accuracy (%)', color=TICK)
ax.set_title('Per-SKU Accuracy (PlantVillage test)', color='white', fontweight='bold')
ax.tick_params(colors=TICK)
for i,v in enumerate(accs):
    ax.text(v+0.5, i, f'{v:.1f}%', va='center', color='white', fontsize=9)
plt.tight_layout()
plt.savefig(ASSETS_DIR/'10_pv_sku_accuracy.png', dpi=130, bbox_inches='tight', facecolor=BG)
plt.close()
print('Saved: 09_pv_confusion_matrix.png, 10_pv_sku_accuracy.png')

In [ ]:
# CELL 10 — EXPORT MULTI-SKU MODEL
PREFIX = MODELS_DIR / 'agrosight_plantvillage'
best_model.export(str(PREFIX))
best_model.save(str(PREFIX) + '.keras')

conv = tf.lite.TFLiteConverter.from_keras_model(best_model)
# float32 only — fp16 breaks browser tfjs-tflite (zero outputs → fake 33% confidence)
with open(str(PREFIX)+'.tflite','wb') as f:
    f.write(conv.convert())

print(f'Exported: {PREFIX}/')
print(f'           {PREFIX}.keras')
print(f'           {PREFIX}.tflite')

In [ ]:
# CELL 11 — SUMMARY
final = metrics[BEST].get('Accuracy_FT', metrics[BEST]['Accuracy'])
print('='*60)
print('  PLANTVILLAGE MULTI-SKU — SUMMARY')
print('='*60)
print(f'  SKUs       : {len(SKU_MAP)} crops (Grains, Spices, Veg, Fruit)')
print(f'  Train/Test : {len(train_paths)} / {len(test_paths)} images')
print(f'  Defects    : {DEFECT_CLASSES}')
print(f'  Winner     : {BEST}')
print(f'  Accuracy   : {final*100:.2f}%')
print(f'  Beans nb   : AgroSight_CV_Complete.ipynb (pilot — keep both)')
print('='*60)
print('Per-SKU accuracy:')
for k,v in sku_acc.items():
    print(f'  {k:18s} {v:.1f}%')